### Import Packages

In [20]:
import os

import pandas as pd

import zipfile
from pathlib import Path


import numpy as np
import pickle

### Notebook Settings

In [21]:
# Display Settings for Pandas DataFrames
pd.set_option("display.max_rows", None)         # How many rows to display when printing a DataFrame? None shows all rows.
pd.set_option("display.max_columns", None)      # How many columns to display when printing a DataFrame? None shows all columns.
pd.set_option("display.max_info_columns", 100)  # How many rows to display for .info()?
pd.set_option("display.max_info_rows", 1000000) # How many columns to display for .info()?
pd.set_option("display.precision", 2)           # How many decimals to show when printing a DataFrame?

### Load Files


In [22]:
# 1. Define paths
zip_path = "../../data/external/data-pack-2016-17/Knee Replacement Provider 1617.csv.zip"
target_csv = "Knee Replacement Provider 1617.csv"

# 2. Open the zip and read the specific CSV
with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(target_csv) as f:
        df_knee_provider_1617 = pd.read_csv(f)

# Display results
print(f"Successfully loaded {target_csv}")
print(df_knee_provider_1617.info())

Successfully loaded Knee Replacement Provider 1617.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47604 entries, 0 to 47603
Data columns (total 81 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Provider Code                                     47604 non-null  object 
 1   Procedure                                         47604 non-null  object 
 2   Revision Flag                                     47604 non-null  int64  
 3   Year                                              47604 non-null  object 
 4   Age Band                                          47604 non-null  object 
 5   Gender                                            47604 non-null  object 
 6   Pre-Op Q Assisted                                 47604 non-null  int64  
 7   Pre-Op Q Assisted By                              47604 non-null  int64  
 8   Pre-Op Q Symptom Period                  

/var/folders/xk/kg13qzdn74nfkmm57d6yv07m0000gn/T/ipykernel_1091/2990646603.py:8: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_knee_provider_1617 = pd.read_csv(f)


TODO: Merge Files

### Create a Copy

In [23]:
df_knee_provider_1617_copy = df_knee_provider_1617.copy(deep=False)

### Descriptive Statistics - Split into Numerical & String DataFrames

In [24]:
# Create table with numerical data
df_knee_provider_1617_num = df_knee_provider_1617.select_dtypes(include=[np.number])

# Column names (Pandas: df_pd_orig_num.columns.tolist()).
l_df_num_names = df_knee_provider_1617_num.columns.tolist()

print(f"We have {len(l_df_num_names)} numerical variables:\n{l_df_num_names}")

We have 76 numerical variables:
['Revision Flag', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Knee Repl

In [25]:
# Create table with string data
df_knee_provider_1617_str = df_knee_provider_1617.select_dtypes(include=["object"])

# Column names (Pandas: df_pd_orig_num.columns.tolist()).
l_df_str_names = df_knee_provider_1617_str.columns.tolist()

print(f"We have {len(l_df_str_names)} string variables:\n{l_df_str_names}")

We have 5 string variables:
['Provider Code', 'Procedure', 'Year', 'Age Band', 'Gender']


CHECK: Is Gender a String Variable?

In [ ]:
print(
    f"The original data has  {df_knee_provider_1617.shape[1]} columns. The number and string data "
    f"have total of {df_knee_provider_1617_num.shape[1] + df_knee_provider_1617_str.shape[1]} columns."
)

The original data has  81 columns. The number and string data have total of 81 columns.


In [27]:
print(f"The numnber of obeservations: {df_knee_provider_1617.shape[0]}")

The numnber of obeservations: 47604


In [28]:
n_obs = df_knee_provider_1617.shape[0]

### Check Missing Variables

In [30]:
df_knee_provider_1617_missing_data_num = (
    
    pd.DataFrame({

        'data_type':    df_knee_provider_1617.dtypes,
        'missing':      df_knee_provider_1617.isnull().sum()   
    })

    # Add complete%
    .assign(
        complete_pct = lambda x: round(100 * (n_obs - x['missing']) / n_obs, 2)
    )

    # Remove variables that are complete.
    .query("complete_pct < 100")

    # Sort table by data completeness in descending order.
    .sort_values(by = 'complete_pct')
)

print(df_knee_provider_1617_missing_data_num)
print("")

                                                 data_type  missing  \
Knee Replacement EQ VAS_Post-Op Q Predicted        float64     6884   
Knee Replacement EQ 5D Index Post-Op Q Predicted   float64     4935   
Pre-Op Q EQ5D Index                                float64     2667   
Post-Op Q EQ5D Index                               float64     2144   
Knee Replacement OKS Post-Op Q Predicted           float64     1491   
Knee Replacement Post-Op Q Score                   float64      645   
Knee Replacement Pre-Op Q Score                    float64      542   

                                                  complete_pct  
Knee Replacement EQ VAS_Post-Op Q Predicted              85.54  
Knee Replacement EQ 5D Index Post-Op Q Predicted         89.63  
Pre-Op Q EQ5D Index                                      94.40  
Post-Op Q EQ5D Index                                     95.50  
Knee Replacement OKS Post-Op Q Predicted                 96.87  
Knee Replacement Post-Op Q Score         